# Reinforcement Learning Agents for PettingZoo Games

The goal of this project is to implement three artificial intelligence models that use Reinforcement Learning (RL) to play games available on the PettingZoo platform.

## Selected Games:
- **Rock Paper Scissors**
- **Tic Tac Toe**

The project aims to create and train an RL agent capable of effectively competing against various opponents.

## install pettingzoo

In [1]:
!pip install pettingzoo

You should consider upgrading via the 'C:\Users\Hubert\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


## **Classic: Rock Paper Scissors (Papier-Kamień-Nożyce)**

### About game

#### Rules of Rock-Paper-Scissors (RPS)

- The game is played between two players (agent vs. opponent).
- Each player chooses one of three symbols: rock, paper, or scissors.
- The goal of the game is to choose the symbol that beats the opponent's symbol:
  - Rock beats Scissors,
  - Scissors beat Paper,
  - Paper beats Rock.
- The game ends with:
  - A win for the player who chose the symbol that beats the opponent’s symbol,
  - A draw if both players choose the same symbol.

---

#### Agent Reward System

| Situation                  | Agent's Reward |
|----------------------------|----------------|
| Win                        | +1             |
| Loss                       | -1             |
| Draw                       | 0              |

The Q-learning agent updates its Q-values based on the received reward and the predicted future value of the best possible move.

---

#### How the Q-learning Agent Works

The agent creates a Q-table where it stores the Q-value for each possible pair (game state, agent's move). Each game state represents the current choice of the agent and the current state of the game. The agent tries to learn which choices are more beneficial based on past experiences.

---

#### Decision-Making Process

- With probability epsilon, the agent chooses a random move (exploration).
- With probability 1 - epsilon, the agent chooses the move with the highest Q-value (exploitation).
- After each game, the agent updates its Q-table using the reward and the predicted future value of the best possible move.

---

#### Opponent Descriptions

#### **RandomOpponent**
- Chooses a move randomly from the available options (rock, paper, scissors).
- It has no strategy or learning mechanism.

#### **CycleOpponent**
- Chooses moves in a cyclical order (e.g., rock, paper, scissors, rock, paper, scissors, etc.).
- It does not change its choice based on the agent's move.

#### **StatisticalOpponent**
- Analyzes the agent’s move history.
- Chooses the move that the agent has most frequently chosen in similar situations (based on statistics).
- It tries to predict and block the agent’s preferences.

#### **DelayMirrorOpponent**
- Chooses the same move that the agent chose in the previous round, but mirrored (e.g., if the agent chose rock, the opponent chooses paper).

#### **SmartOpponent**
- A Q-learning agent.
- It learns exactly the same way as the main agent but plays as an opponent.
- It is dynamic and adapts to the agent’s strategy, making it the toughest opponent.

---

#### Training Process

The agent's training consists of multiple episodes (e.g., 10,000 games against each opponent). Each episode consists of a series of games where the agent has the opportunity to adjust its strategies.

#### Training Features:
- Opponents are chosen randomly with balance to ensure diverse experiences.
- Epsilon gradually decreases, allowing the agent to increasingly exploit the learned strategies.

---

#### Agent Testing

After training is complete, the agent is tested against each opponent in 1,000 games. Testing occurs without further learning, meaning the agent does not change its strategies during the tests (exploration is disabled). The goal is to evaluate the effectiveness of the learned strategies.

### Code and output

In [45]:
import numpy as np
import random
from pettingzoo.utils.env import ParallelEnv
from gymnasium import spaces

# Definition of the "rock, paper, scissors" game environment class
class RockPaperScissorsEnv(ParallelEnv):
    metadata = {"render_modes": ["human"], "name": "rock_paper_scissors_v0"}

    # Environment initialization
    def __init__(self, max_rounds=1000):
        # List of available agents
        self.possible_agents = ["player_0", "player_1"]
        # Action space for each agent (3 possible actions: rock, paper, scissors)
        self.action_spaces = {a: spaces.Discrete(3) for a in self.possible_agents}
        # Observation space (6 values for each player: 3 moves in the past from both players)
        self.observation_spaces = {a: spaces.MultiDiscrete([3]*6) for a in self.possible_agents}
        self.max_rounds = max_rounds  # Maximum number of rounds
        self.has_reset = False  # Flag to check if reset has been called

    # Function to reset the environment state at the start of the game
    def reset(self, seed=None, options=None):
        self.agents = self.possible_agents[:]  # Initialize agents
        self.round_count = 0  # Round counter
        # History of player moves, initially no moves have been made
        self.history = {a: [0,0,0] for a in self.agents}
        self.has_reset = True  # Set the reset flag
        # Return the observation for each player (history of the last 3 moves of both players)
        return {a: np.array(self._get_obs(a), dtype=int) for a in self.agents}

    # Function to execute one step in the game
    def step(self, actions):
        assert self.has_reset, "reset() before step()"  # Check if reset was called before step()
        a0, a1 = actions["player_0"], actions["player_1"]  # Assign player actions
        # Calculate rewards (results) for both players
        r0, r1 = self._reward(a0, a1), self._reward(a1, a0)
        # Update move history
        self.history["player_0"].pop(0); self.history["player_0"].append(a0)
        self.history["player_1"].pop(0); self.history["player_1"].append(a1)
        self.round_count += 1  # Increment the round counter
        # Generate new observations for players
        obs = {a: np.array(self._get_obs(a), dtype=int) for a in self.agents}
        # Check if the game should end (after reaching the maximum number of rounds)
        done = self.round_count >= self.max_rounds
        # Set the game-ending flag for all players
        dones = {a: done for a in self.agents}
        truncs = {a: False for a in self.agents}  # "Truncation" flags - not used in this game
        # Return rewards and game-end information
        rewards = {"player_0": r0, "player_1": r1}
        infos = {a: {} for a in self.agents}  # Empty info structure for agents
        return obs, rewards, dones, truncs, infos

    # Function to return the observation for a given agent
    def _get_obs(self, agent):
        if agent == "player_0":
            # Observation for player 0: their history and player 1's move history
            return self.history["player_0"] + self.history["player_1"]
        else:
            # Observation for player 1: their history and player 0's move history
            return self.history["player_1"] + self.history["player_0"]

    # Function to return the reward based on the game result
    @staticmethod
    def _reward(a, b):
        if a == b:
            return 0  # Draw
        return -1 if (a+1)%3 == b else 1  # -1 if player a loses, 1 if they win

    # Functions for rendering and closing the environment (not used in this implementation)
    def render(self): pass
    def close(self): pass

# Class representing an opponent that chooses a move randomly
class RandomOpponent:
    def choose_action(self, obs): return np.random.choice(3)

# Class representing an opponent that cyclically changes their moves
class CycleOpponent:
    def __init__(self): self.next_move = 0  # Initial move
    def choose_action(self, obs):
        m = self.next_move
        self.next_move = (m+1)%3  # Next move (cyclically: 0 -> 1 -> 2 -> 0)
        return m

# Class representing an opponent that chooses a move based on the opponent's move statistics
class StatisticalOpponent:
    def __init__(self): self.counts = [0,0,0]  # Opponent's move counter
    def choose_action(self, obs):
        for mv in obs[3:]: self.counts[mv] += 1  # Count the opponent's moves
        most = int(np.argmax(self.counts))  # Choose the most frequent move
        return (most+1)%3  # Choose the move that will counter the opponent's most frequent move

# Class representing an opponent that copies the player's previous move
class DelayMirrorOpponent:
    def __init__(self): self.last_agent = np.random.choice(3)  # Random initial move
    def choose_action(self, obs):
        act = self.last_agent  # Copy previous move
        self.last_agent = int(obs[5])  # Remember the player's move
        return act

# Class representing a Q-learning agent
class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_decay=0.9999):
        self.alpha, self.gamma = alpha, gamma  # Q-learning coefficients
        self.epsilon, self.epsilon_decay = epsilon, epsilon_decay  # Exploration parameters
        self.Q = np.zeros((3**6,3), dtype=float)  # Q-table initialized to zero (for 3^6 states and 3 actions)

    # Function to transform observation into a state identifier
    def _state_id(self, obs):
        s = 0
        for i,v in enumerate(obs):
            s += int(v)*(3**i)  # Transforming the state into a unique identifier
        return s

    # Function to choose an action by the agent (exploration or exploitation)
    def choose_action(self, obs, explore=True):
        s = self._state_id(obs)  # Calculate state identifier
        if explore and np.random.rand() < self.epsilon:
            return np.random.choice(3)  # Exploration - random choice
        return int(np.argmax(self.Q[s]))  # Exploitation - choose the best action based on Q

    # Function to update the agent's Q-values based on the executed action and received reward
    def update(self, obs, a, r, next_obs):
        s, ns = self._state_id(obs), self._state_id(next_obs)  # State identifiers before and after action
        td = r + self.gamma*np.max(self.Q[ns]) - self.Q[s, a]  # Temporal Difference (TD) calculation
        self.Q[s, a] += self.alpha * td  # Update Q-values
        self.epsilon *= self.epsilon_decay  # Decrease epsilon to reduce exploration

# Class representing a Q-learning-based agent
class SmartOpponent(QLearningAgent):
    pass

# Function to train the agent in "interleaved" mode (alternating games with different opponents)
def train_interleaved(agent, opponents, total_games_per_opp=10000, block_size=1000):
    remaining = {op: total_games_per_opp for op in opponents}  # Number of games remaining for each opponent
    results = []  # List of training results
    total_round = 1  # Round number
    opponent_rounds = {op: 1 for op in opponents}  # Round counter for each opponent

    # Training loop
    while any(remaining[op] > 0 for op in opponents):
        available = [op for op in opponents if remaining[op] > 0]  # List of opponents with remaining games
        opp = random.choice(available)  # Randomly choose an opponent
        n = min(block_size, remaining[opp])  # Number of games to play in this iteration

        env = RockPaperScissorsEnv(max_rounds=n)  # Initialize the environment
        obs = env.reset()  # Reset the environment
        opp_round = opponent_rounds[opp]  # Opponent's round counter
        wins, draws, losses = 0, 0, 0  # Game result counters

        # Gameplay loop
        for _ in range(n):
            a0 = agent.choose_action(obs["player_0"], explore=True)  # Agent's action choice
            a1 = opp.choose_action(obs["player_1"])  # Opponent's action choice
            obs2, rewards, dones, truncs, infos = env.step({
                "player_0": a0, "player_1": a1  # Take a step in the game
            })
            agent.update(obs["player_0"], a0, rewards["player_0"], obs2["player_0"])  # Update agent
            if isinstance(opp, QLearningAgent):
                opp.update(obs["player_1"], a1, rewards["player_1"], obs2["player_1"])  # Update opponent (if Q-learning)
            obs = obs2  # Update observations
            # Count results
            if rewards["player_0"] == 1:
                wins += 1
            elif rewards["player_0"] == 0:
                draws += 1
            else:
                losses += 1
            opp_round += 1
            if opp_round > 10:  # Reset opponent's round counter every 10 games
                opp_round = 1

        # Save the results
        results.append({
            "opponent": opp.__class__.__name__,
            "total_round": total_round,
            "opponent_round": opponent_rounds[opp],
            "win": wins,
            "draw": draws,
            "loss": losses
        })

        total_round += 1  # Increment round number
        remaining[opp] -= n  # Decrease remaining games
        opponent_rounds[opp] += 1  # Increment opponent's round count

    # Display training results
    headers = ["OPPONENT", "TOTAL ROUND", "OPPONENT ROUND", "WIN", "DRAW", "LOSS"]
    col_widths = {
        "OPPONENT": max(len(r["opponent"]) for r in results + [{"opponent": "OPPONENT"}]),
        "TOTAL ROUND": max(len(str(r["total_round"])) for r in results + [{"total_round": "TOTAL ROUND"}]),
        "OPPONENT ROUND": max(len(str(r["opponent_round"])) for r in results + [{"opponent_round": "OPPONENT ROUND"}]),
        "WIN": max(len(str(r["win"])) for r in results + [{"win": "WIN"}]),
        "DRAW": max(len(str(r["draw"])) for r in results + [{"draw": "DRAW"}]),
        "LOSS": max(len(str(r["loss"])) for r in results + [{"loss": "LOSS"}]),
    }

    separator = "+".join("-" * (col_widths[h] + 2) for h in headers)
    separator = f"+{separator}+"
    header_row = "| " + " | ".join(f"{h:^{col_widths[h]}}" for h in headers) + " |"

    print("\nTRAINING RESULTS:")  # Display the results
    print(separator)
    print(header_row)
    print(separator)
    for r in results:
        row = "| " + " | ".join([  # Format the result row
            f"{r['opponent']:^{col_widths['OPPONENT']}}",
            f"{r['total_round']:^{col_widths['TOTAL ROUND']}}",
            f"{r['opponent_round']:^{col_widths['OPPONENT ROUND']}}",
            f"{r['win']:^{col_widths['WIN']}}",
            f"{r['draw']:^{col_widths['DRAW']}}",
            f"{r['loss']:^{col_widths['LOSS']}}"
        ]) + " |"
        print(row)
    print(separator)

    return agent  # Return the agent after training

# Function to test the agent's performance against opponents
def test_sequential(agent, opponents, test_games=1000):
    test_results = []  # List of test results
    for opp in opponents:  # Test against all opponents
        wins = ties = losses = 0  # Result counters
        env = RockPaperScissorsEnv(max_rounds=test_games)  # Initialize the environment
        obs = env.reset()  # Reset the environment
        for _ in range(test_games):  # Play the test games
            a0 = agent.choose_action(obs["player_0"], explore=False)  # Agent's action
            if isinstance(opp, QLearningAgent):
                a1 = opp.choose_action(obs["player_1"], explore=False)  # Opponent's action (Q-learning)
            else:
                a1 = opp.choose_action(obs["player_1"])  # Opponent's action (other methods)
            obs2, rewards, dones, truncs, infos = env.step({
                "player_0": a0, "player_1": a1  # Take a step in the game
            })
            r = rewards["player_0"]  # Agent's reward
            # Count results
            if r == 1:
                wins += 1
            elif r == 0:
                ties += 1
            else:
                losses += 1
            obs = obs2  # Update observations

        # Save the test results
        test_results.append({
            "opponent": opp.__class__.__name__,
            "wins": wins,
            "draws": ties,
            "losses": losses
        })

    # Display test results
    headers = ["OPPONENT", "WIN", "DRAW", "LOSS"]
    col_widths = {
        "OPPONENT": max(len(r["opponent"]) for r in test_results + [{"opponent": "OPPONENT"}]),
        "WIN": max(len(str(r["wins"])) for r in test_results + [{"wins": "WIN"}]),
        "DRAW": max(len(str(r["draws"])) for r in test_results + [{"draws": "DRAW"}]),
        "LOSS": max(len(str(r["losses"])) for r in test_results + [{"losses": "LOSS"}]),
    }

    separator = "+".join("-" * (col_widths[h] + 2) for h in headers)
    separator = f"+{separator}+"
    header_row = "| " + " | ".join(f"{h:^{col_widths[h]}}" for h in headers) + " |"

    print("\nTEST RESULTS:")  # Display the test results
    print(separator)
    print(header_row)
    print(separator)
    for r in test_results:
        row = "| " + " | ".join([  # Format the result row
            f"{r['opponent']:^{col_widths['OPPONENT']}}",
            f"{r['wins']:^{col_widths['WIN']}}",
            f"{r['draws']:^{col_widths['DRAW']}}",
            f"{r['losses']:^{col_widths['LOSS']}}"
        ]) + " |"
        print(row)
    print(separator)

if __name__ == "__main__":
    # List of opponents the agent will play against
    opponents = [
        RandomOpponent(),
        CycleOpponent(),
        StatisticalOpponent(),
        DelayMirrorOpponent(),
        SmartOpponent(),
    ]

    agent = QLearningAgent()  # Initialize the agent
    # Train the agent against opponents
    agent = train_interleaved(agent, opponents, total_games_per_opp=10000, block_size=1000)
    # Test the agent against various opponents
    test_sequential(agent, opponents, test_games=1000)


TRAINING RESULTS:
+---------------------+-------------+----------------+-----+------+------+
|      OPPONENT       | TOTAL ROUND | OPPONENT ROUND | WIN | DRAW | LOSS |
+---------------------+-------------+----------------+-----+------+------+
| DelayMirrorOpponent |      1      |       1        | 352 | 314  | 334  |
| StatisticalOpponent |      2      |       1        | 405 | 309  | 286  |
| StatisticalOpponent |      3      |       2        | 438 | 282  | 280  |
|    SmartOpponent    |      4      |       1        | 370 | 315  | 315  |
|   RandomOpponent    |      5      |       1        | 309 | 320  | 371  |
|    SmartOpponent    |      6      |       2        | 304 | 365  | 331  |
| StatisticalOpponent |      7      |       3        | 533 | 234  | 233  |
|    SmartOpponent    |      8      |       3        | 297 | 366  | 337  |
|   RandomOpponent    |      9      |       2        | 335 | 320  | 345  |
| StatisticalOpponent |     10      |       4        | 574 | 237  | 189  |
| Dela

### Summary

The agent demonstrated high effectiveness in games against various opponents, with the exception of the RandomOpponent, which was expected due to the unpredictability of its behavior (the agent couldn't learn any patterns). The best results were achieved in matches against the cyclical opponent, where the number of wins was decisive. In confrontations with more challenging opponents, such as the StatisticalOpponent and DelayMirrorOpponent, the agent still maintained an advantage, although it had to deal with greater difficulties. A high number of draws occurred mainly in the match against the SmartOpponent, which shows that the agent faced a greater challenge – this is logical given the nature of the opponent.

In summary, the agent demonstrated good skills during the tests, proving that it can adapt its strategies to different types of opponents, even when faced with more advanced tactics.

## **Classic: Classic: Tic Tac Toe (Kółko i Krzyżyk)**

### About game

#### Rules of Tic-Tac-Toe

- The board consists of 9 fields (3x3).
- Players take turns to make moves, placing either "X" (player 1) or "O" (player 2).
- The goal of the game is to arrange three of your symbols in a line (horizontal, vertical, or diagonal).
- The game ends with:
  - A win for one of the players.
  - A draw (if all fields are filled and no player has won).

---

#### Agent Reward System

| Situation                  | Agent Reward |
|----------------------------|--------------|
| Win                        | +1           |
| Loss                       | -1           |
| Draw                       | 0            |

The Q-learning agent updates its Q-values (expected rewards) based on the received reward and the predicted future value of the best possible move.

---

#### How the Q-learning Agent Works

The agent builds a Q-table, in which it records the Q-value for each pair (board state, possible action) —the predicted future reward. Based on this table, the agent learns which moves are more profitable.

#### Decision-making process:

- With probability epsilon, the agent chooses a random move (exploration).
- With probability 1 - epsilon, the agent chooses the move with the highest Q-value (exploitation).
- After each game, the agent updates the Q-values based on the received reward and the predicted future value of the best possible move.

---

#### Opponent Descriptions

#### **RandomOpponent**
- Chooses a random move from the available options.
- Does not learn or predict the opponent's behavior.

#### **CycleOpponent**
- Makes moves in a cyclic, fixed order (from 0 to 8).
- Does not respond to the agent’s actions.

#### **StatisticalOpponent**
- Analyzes the agent’s move history.
- Predicts which move the agent will most frequently choose and tries to block it.
- Learns based on the statistical occurrence of moves.

#### **MirrorOpponent**
- Attempts to repeat the agent's last move, but in a mirrored version of the board.
- If the agent's move cannot be mirrored, it makes a random move.

#### **SmartOpponent**
- Is a Q-learning agent.
- Learns exactly the same way as the main agent, but plays as an opponent.
- Considered the toughest opponent – it dynamically adapts to the agent's strategy.

---

#### Training Process

Training consists of many episodes (10,000 per opponent), divided into blocks of 1,000 games. In each iteration, the agent plays against a randomly chosen opponent, learning during the game.

#### Training Features:
- In each round, an opponent is chosen randomly, maintaining balance.
- Epsilon gradually decreases, allowing the agent to increasingly exploit known strategies.
- Opponents are not present simultaneously – the agent plays against one opponent at a time.

---

#### Agent Testing

After training, the agent is tested against each opponent in 1,000 test games, without further learning (exploration is turned off). The goal is to assess the effectiveness of the learned strategies.

### Code and output

In [44]:
import random
import numpy as np
import prettytable as pt
from pettingzoo import AECEnv
from gym.spaces import Discrete

# Helper function: checks if there is a winner
def check_winner_state(state):
    # Definition of all possible winning lines (vertical, horizontal, diagonal)
    lines = [
        (0, 1, 2), (3, 4, 5), (6, 7, 8),
        (0, 3, 6), (1, 4, 7), (2, 5, 8),
        (0, 4, 8), (2, 4, 6)
    ]
    # Check if any line is full (all players on the line have the same symbol)
    for (i, j, k) in lines:
        if state[i] == state[j] == state[k] != 0:
            return state[i]  # Return the winner (1 for player 1 or -1 for player 2)
    # If all fields are filled and there is no winner, it's a draw
    if all(v != 0 for v in state):
        return 0
    return None  # If there is no winner yet, return None

# Game environment - TicTacToeEnv class inherits from AECEnv, allowing interaction with agents
class TicTacToeEnv(AECEnv):
    def __init__(self):
        # Game state: list of 9 elements representing the board (0 means empty, 1 - X, -1 - O)
        self.state = [0] * 9
        self.agents = ['player_1', 'player_2']  # List of agents
        self.agent_order = self.agents  # Order of agents
        self.current_agent = self.agent_order[0]  # First agent (player 1)
        self.done = False  # Game is not over yet

    # Reset function for the game state (clears the board, sets the first move for the player)
    def reset(self):
        self.state = [0] * 9  # Reset the board
        self.done = False  # The game is still ongoing
        self.current_agent = self.agent_order[0]  # Player 1 starts
        return {self.current_agent: self.state.copy()}  # Return the board state for the current agent

    # Function returning legal actions (empty fields)
    def legal_actions(self):
        return [i for i, v in enumerate(self.state) if v == 0]

    # Function to make a move in the game
    def step(self, action):
        if self.done:
            raise Exception("Game is over. Reset the environment to start a new game.")

        agent_idx = self.agent_order.index(self.current_agent)  # Index of the current player
        player = 1 if agent_idx == 0 else -1  # Player 1 (X) is 1, Player 2 (O) is -1
        self.state[action] = player  # Perform the move on the board

        winner = check_winner_state(self.state)  # Check if there is a winner
        if winner is not None:
            self.done = True  # Game is over
            reward = 1 if winner == 1 else -1 if winner == -1 else 0  # Reward: 1 for a win, -1 for a loss, 0 for a draw
        else:
            reward = 0
            self.done = all(v != 0 for v in self.state)  # Game ends in a draw if the board is full

        # Switch to the next player
        self.current_agent = self.agent_order[1 - agent_idx]
        return {self.current_agent: self.state.copy()}, reward, self.done, {}

    # Function to render the game state as text
    def render(self):
        table = ""
        for i in range(3):
            table += f"{self.state[3 * i]} | {self.state[3 * i + 1]} | {self.state[3 * i + 2]}\n"
            if i < 2:
                table += "--------\n"
        return table

    # Function to return the action space (all board positions)
    def action_spaces(self):
        return {agent: Discrete(9) for agent in self.agents}

    # Function to return the observation space (all board positions)
    def observation_spaces(self):
        return {agent: Discrete(9) for agent in self.agents}

# Q-learning agent class
class QLearningAgent:
    def __init__(self, lr=0.1, gamma=0.95, epsilon=1.0, decay=0.9995, min_eps=0.1):
        self.q = {}  # Dictionary to store Q values (key: tuple (state, action), value: Q)
        self.lr = lr  # Learning rate
        self.gamma = gamma  # Discount factor for future rewards
        self.epsilon = epsilon  # Probability of exploration (random move)
        self.decay = decay  # Decay factor for epsilon
        self.min_eps = min_eps  # Minimum value for epsilon

    def get(self, state, action):
        # Returns the Q value for a given state and action (0 if not present)
        return self.q.get((tuple(state), action), 0.0)

    def choose_action(self, state, legal, training=True):
        # Chooses an action based on the epsilon-greedy strategy
        if training and random.random() < self.epsilon:  # Exploration (random move)
            return random.choice(legal)
        # Exploitation (choosing the action with the best Q value)
        qs = [self.get(state, a) for a in legal]
        max_q = max(qs)
        return random.choice([a for a, q in zip(legal, qs) if q == max_q])

    def learn(self, prev, action, reward, next_st, done, legal_next):
        # Updates the Q value based on the received reward and the predicted future value
        prev_key = tuple(prev)
        target = reward if done else reward + self.gamma * max([self.get(next_st, a) for a in legal_next])
        current = self.get(prev, action)
        self.q[(prev_key, action)] = current + self.lr * (target - current)

        # Decrease epsilon (reduce exploration)
        if self.epsilon > self.min_eps:
            self.epsilon *= self.decay

# Opponents (random and strategic opponents)
class RandomOpponent:
    def choose_action(self, state, legal):
        return random.choice(legal)  # Random move

class CycleOpponent:
    def __init__(self):
        self.order = list(range(9))  # Sequence of moves
        self.idx = 0

    def choose_action(self, state, legal):
        for _ in range(9):
            move = self.order[self.idx % 9]  # Cyclic move
            self.idx += 1
            if move in legal:
                return move
        return random.choice(legal)  # If cycle is blocked, random choice

class StatisticalOpponent:
    def __init__(self):
        self.counts = [0] * 9  # Number of times each position was chosen

    def choose_action(self, state, legal):
        if sum(self.counts) == 0:
            return random.choice(legal)  # If not played yet, random choice
        pred = max(range(9), key=lambda i: self.counts[i])  # Choose the most frequently blocked position
        return pred if pred in legal else random.choice(legal)  # If predicted is unavailable, random choice

    def update(self, action):
        self.counts[action] += 1  # Increase the counter for the chosen move

class SmartOpponent(QLearningAgent):
    def __init__(self):
        super().__init__(epsilon=1.0, decay=0.9995, min_eps=0.1, lr=0.1, gamma=0.95)  # Q-learning agent

class MirrorOpponent:
    def __init__(self):
        self.last_agent_action = None  # Last agent's move
        self.mirror_map = {  # Mirror map
            0: 8, 1: 7, 2: 6,
            3: 5, 4: 4, 5: 3,
            6: 2, 7: 1, 8: 0
        }

    def choose_action(self, state, legal):
        if self.last_agent_action is not None:
            mirrored = self.mirror_map[self.last_agent_action]  # Mirror the last move
            if mirrored in legal:
                return mirrored  # Return mirrored move if legal
        return random.choice(legal)  # If cannot mirror, choose randomly

    def update(self, action):
        self.last_agent_action = action  # Remember the last move

# Gameplay function between the agent and the opponent
def play_episode(env, agent, opponent, training=True):
    state = env.reset()  # Reset the environment
    state = state['player_1']  # Get the state for player 1
    done = False

    # Update the opponent
    if isinstance(opponent, (MirrorOpponent, StatisticalOpponent)):
        opponent.update(None if isinstance(opponent, MirrorOpponent) else 0)

    # Gameplay loop
    while not done:
        legal = env.legal_actions()  # Possible legal moves
        a = agent.choose_action(state, legal, training)  # Agent chooses a move
        step_out, reward, done, _ = env.step(a)  # Agent makes a move
        ns = step_out['player_2']  # New state for the opponent

        if training:
            agent.learn(state, a, reward, ns, done, env.legal_actions())  # Agent learns

        if isinstance(opponent, (StatisticalOpponent, MirrorOpponent)):
            opponent.update(a)  # Update the opponent

        state = ns  # Update state
        if done:
            return reward  # Return the result

        # The opponent makes a move
        legal = env.legal_actions()
        o = opponent.choose_action(state, legal)  # Opponent chooses a move
        step_out, r_opp, done, _ = env.step(o)  # Opponent makes a move
        ns = step_out['player_1']  # New state for the agent

        if training and isinstance(opponent, QLearningAgent):
            opponent.learn(state, o, r_opp, ns, done, env.legal_actions())  # Opponent learns

        state = ns  # Update the state
        if done:
            return -1  # Return the result if the game is over

    return 0  # Draw

# Training function for the agent with opponents
def fair_mixed_train(agent, opponent_classes, episodes_per_opponent=10000, block=1000):
    env = TicTacToeEnv()  # Initialize the game environment
    block_counter = 0
    results = []
    all_opponents = []
    for cls in opponent_classes:
        all_opponents.extend([cls] * (episodes_per_opponent // block))  # Prepare a list of opponents
    random.shuffle(all_opponents)  # Shuffle the opponents

    opponent_round_counter = {cls.__name__: 0 for cls in opponent_classes}  # Round counter for each opponent

    for opp_cls in all_opponents:
        wins = draws = losses = 0  # Counters for wins, draws, and losses for a given opponent
        for _ in range(block):
            opp = opp_cls()  # Create an instance of the opponent
            result = play_episode(env, agent, opp, training=True)  # Play against the opponent
            if result == 1: wins += 1
            elif result == 0: draws += 1
            else: losses += 1

        block_counter += 1
        opponent_name = opp_cls.__name__
        opponent_round_counter[opponent_name] += 1
        results.append([  # Save results for the given opponent
            opponent_name,
            block_counter,
            opponent_round_counter[opponent_name],
            wins,
            draws,
            losses
        ])
    return results  # Return training results

# Function to test the agent against a specific opponent
def test_against(agent, opponent_class, episodes=1000):
    env = TicTacToeEnv()
    wins = draws = losses = 0
    for _ in range(episodes):
        opp = opponent_class()  # Create an instance of the opponent
        result = play_episode(env, agent, opp, training=False)  # Test the agent
        if result == 1: wins += 1
        elif result == 0: draws += 1
        else: losses += 1
    return wins, draws, losses  # Return test results

# List of opponents
opponent_classes = [RandomOpponent, CycleOpponent, StatisticalOpponent, MirrorOpponent, SmartOpponent]
agent = QLearningAgent()  # Create a Q-learning agent

# Training the agent
train_results = fair_mixed_train(agent, opponent_classes, episodes_per_opponent=10000, block=1000)

# Training results
train_table = pt.PrettyTable()
train_table.field_names = ["OPPONENT", "TOTAL ROUND", "OPPONENT ROUND", "WIN", "DRAW", "LOSS"]
for row in train_results:
    train_table.add_row(row)  # Add results to the table
print("\nTRAINING RESULTS:")
print(train_table)

# Test results
print("\nTEST RESULTS:")
test_table = pt.PrettyTable()
test_table.field_names = ["OPPONENT", "WIN", "DRAW", "LOSS"]
for cls in opponent_classes:
    w, d, l = test_against(agent, cls)  # Test the agent against all opponents
    test_table.add_row([cls.__name__, w, d, l])  # Add results to the table
print(test_table)


TRAINING RESULTS:
+---------------------+-------------+----------------+-----+------+------+
|       OPPONENT      | TOTAL ROUND | OPPONENT ROUND | WIN | DRAW | LOSS |
+---------------------+-------------+----------------+-----+------+------+
| StatisticalOpponent |      1      |       1        | 620 | 118  | 262  |
|    SmartOpponent    |      2      |       1        | 639 | 112  | 249  |
|    RandomOpponent   |      3      |       1        | 699 |  92  | 209  |
|    SmartOpponent    |      4      |       2        | 713 |  83  | 204  |
|    RandomOpponent   |      5      |       2        | 691 |  94  | 215  |
|    CycleOpponent    |      6      |       1        | 739 |  12  | 249  |
|    CycleOpponent    |      7      |       2        | 749 |  17  | 234  |
| StatisticalOpponent |      8      |       2        | 769 |  73  | 158  |
|    MirrorOpponent   |      9      |       1        | 545 | 329  | 126  |
|    SmartOpponent    |      10     |       3        | 757 |  78  | 165  |
|    M

### Summary

The agent demonstrated high effectiveness, winning the vast majority of games against various types of opponents. The best results were achieved in matches against RandomOpponent and StatisticalOpponent, where the differences in the number of wins were significant. Opponents such as MirrorOpponent and SmartOpponent were more challenging, but the agent still maintained an advantage, albeit not without difficulties. The number of draws was higher in matches against more advanced opponents like CycleOpponent and MirrorOpponent, suggesting that the agent encountered greater challenges in these cases.

In summary, the agent performed well in the tests, proving that it had learned effective strategies for winning in Tic-Tac-Toe, especially against opponents based on randomness and simple algorithms.